# PharmKG → Knowledge Graph (KG) Builder

**Source:** PharmKG-180k (`raw_PharmKG-180k.csv`)  
**Species:** *Homo sapiens*

## Relation types processed

| Relation | Head ID | Tail ID |
|---|---|---|
| Gene_Gene | NCBI Symbol (via fullname + synonym map) | NCBI Symbol |
| Gene_Disease | NCBI Symbol | DOID (DO → MeSH ) |
| Gene_Chemical | NCBI Symbol | PubChem CID |
| Chemical_Gene | PubChem CID | NCBI Symbol |
| Chemical_Chemical | PubChem CID | PubChem CID |
| Chemical_Disease | PubChem CID | DOID (DO → MeSH ) |
| Disease_Gene | DOID (DO → MeSH cascade) | NCBI Symbol |
| Disease_Disease | DOID (DO → MeSH cascade) | DOID |
| Disease_Chemical | DOID | PubChem CID |

## Output files
```
PharmKG/   PharmKG_Gene_Gene.csv
PharmKG/   PharmKG_Gene_Disease.csv
PharmKG/   PharmKG_Gene_Chemical.csv
PharmKG/   PharmKG_Chemical_Gene.csv
PharmKG/   PharmKG_Chemical_Chemical.csv
PharmKG/   PharmKG_Chemical_Disease.csv
PharmKG/   PharmKG_Disease_Gene.csv
PharmKG/   PharmKG_Disease_Disease.csv
PharmKG/   PharmKG_Disease_Chemical.csv
```



---
## 0 · Configuration — edit ONLY these two lines

In [1]:
import os
import numpy as np
import pandas as pd
from glob import glob

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/"

# ── Derived input paths (do not edit below this line) ────────────────────────
PUBCHEM_PKL_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl")
PUBCHEM_DB_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
PUBCHEM_SYN_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
DO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
MESH2DOID_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv")
MESH_COMB_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_Combined.csv")
MESH_SUPP_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/Mesh_Supplementary_Info.csv")
ENS2NCBI_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
PHARMKG_PATH       = os.path.join(BASE_PATH, "pharmkg/raw_PharmKG-180k.csv")

os.makedirs(OUT_PATH, exist_ok=True)

# Standard KG schema column order
DESIRED_COLS = [
    "Head", "Relation", "Tail", "Head_type", "Relation_type", "Tail_type",
    "Source", "KG_Source", "Head_ID", "Head_Detail_name", "Head_ENS",
    "Tail_name", "Tail_DOID_Name", "Tail_ENS", "Tail_Detail_name",
    "Tail_Alias_mapped", "Head_ID_IS", "Tail_ID_IS",
    "Head_Orignal", "Tail_Orignal", "PubMed_ID", "Sentence_tokenized"
]

print("Paths configured.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/


---
## 1 · Load reference dictionaries

In [2]:
# ── 1a. PubChem SMILES -> CID ─────────────────────────────────────────────────
Pubchem = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_Smile_CID_Dict = dict(zip(Pubchem['PUBCHEM_SMILES'], Pubchem['PUBCHEM_COMPOUND_CID']))
del Pubchem
print(f"PubChem SMILES -> CID: {len(Pubchem_Smile_CID_Dict):,}")

PubChem SMILES -> CID: 119,125,801


In [3]:
# ── 1b. DrugBank -> PubChem CID and ChEBI -> PubChem CID ─────────────────────
pubchem = pd.read_csv(PUBCHEM_DB_PATH)
pubchem_DB    = pubchem[~pubchem['DRUGBANK_ID'].isna()]
pubchem_CHEBI = pubchem[~pubchem['CHEBI_ID'].isna()]
DB2PC_dict    = dict(zip(pubchem_DB['DRUGBANK_ID'],  pubchem_DB['ID']))
Chebi2PC_dict = dict(zip(pubchem_CHEBI['CHEBI_ID'], pubchem_CHEBI['ID']))
del pubchem
print(f"DrugBank -> PubChem: {len(DB2PC_dict):,} | ChEBI -> PubChem: {len(Chebi2PC_dict):,}")

/tmp/ipykernel_1550168/2583565483.py:2: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem = pd.read_csv(PUBCHEM_DB_PATH)


DrugBank -> PubChem: 10,702 | ChEBI -> PubChem: 177,680


In [4]:
# ── 1c. PubChem synonym -> CID (case-insensitive) ────────────────────────────
# NameError when MESH sections tried to use it. Loaded ONCE here, never deleted.
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict       = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))
# Pre-build lowercase version for case-insensitive lookups throughout
Pubchem_Syn_fil_dict_lower = {str(k).lower(): v for k, v in Pubchem_Syn_fil_dict.items()}
del Pubchem_Syn_fil
print(f"PubChem synonyms: {len(Pubchem_Syn_fil_dict_lower):,}")

PubChem synonyms: 102,877,659


In [5]:
# ── 1d. Disease Ontology (DO) ─────────────────────────────────────────────────
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))   # DOID -> label
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))   # DOID -> xrefs
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))      # label -> DOID
print(f"DO entries: {len(DOID_disname_DOID_dict):,}")

DO entries: 11,787


In [6]:
# ── 1e. MeSH -> DOID crosswalk ────────────────────────────────────────────────
Mesh2DOID = pd.read_csv(MESH2DOID_PATH, sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict    = dict(zip(Mesh2DOID['xrefs'],  Mesh2DOID['id']))     # MeSH xref -> DOID
Meshname2DOID_dict = dict(zip(Mesh2DOID['label'], Mesh2DOID['xrefs']))  # MeSH label -> MeSH xref
print(f"MeSH -> DOID: {len(Mesh2DOID_dict):,} | MeSH name -> xref: {len(Meshname2DOID_dict):,}")

MeSH -> DOID: 3,687 | MeSH name -> xref: 3,973


In [7]:
# ── 1f. MeSH combined and supplementary (ID -> name) ─────────────────────────
MESH = pd.read_csv(MESH_COMB_PATH)
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))
# Note: MESH PubChem mapping removed — Pubchem_Syn_fil_dict is for chemicals,
# not appropriate for mapping disease names

Mesh_supp = pd.read_csv(MESH_SUPP_PATH)
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))
print(f"MeSH combined: {len(MESH_dict):,} | MeSH supplementary: {len(Mesh_supp_dict):,}")

MeSH combined: 38,520 | MeSH supplementary: 324,150


In [8]:
# ── 1g. ENSEMBL <-> NCBI gene symbol crosswalk ───────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [9]:
# ── 1h. NCBI Human gene_info ──────────────────────────────────────────────────
# PharmKG stores gene names as full descriptions (e.g. 'peroxisome proliferator activated receptor gamma')
# NCBI_fullname_2_SYMBOL_dict maps these back to canonical symbols
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID']       = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict             = dict(zip(NCBI_ALL_GENE['Symbol'],      NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict           = dict(zip(NCBI_ALL_GENE['GeneID'],      NCBI_ALL_GENE['Symbol']))
NCBI_fullname_2_SYMBOL_dict        = dict(zip(NCBI_ALL_GENE['description'], NCBI_ALL_GENE['Symbol']))
print(f"NCBI Human genes: {len(NCBI_ALL_GENEname_dict):,}")

NCBI Human genes: 193,352


In [10]:
# ── 1i. Gene synonym map — built ONCE globally ────────────────────────────────
# Built once here; all sections reuse this shared dict.
synonym_map = {}
for _, row in NCBI_ALL_GENE.iterrows():
    for syn in row['Synonyms'].split('|'):
        synonym_map[syn.strip()] = row['Symbol']
print(f"Gene synonym map: {len(synonym_map):,} aliases")

Gene synonym map: 69,564 aliases


---
## 2 · Load and split PharmKG-180k

In [11]:
# ── 2a. Load raw PharmKG ─────────────────────────────────────────────────────
KG_file = pd.read_csv(PHARMKG_PATH)
print(f"Raw PharmKG: {len(KG_file):,} rows")
print("Entity types:", set(KG_file['Entity1_type']))

Raw PharmKG: 1,093,236 rows
Entity types: {'gene', 'disease', 'Disease', 'chemical', 'Gene', 'Chemical'}


In [12]:
# ── 2b. Standardise entity types and build Relation string ───────────────────
# Fixed: lowercase Entity types; lowercase Entity2_name (compound/disease names); title-case Relation
KG_file['Entity1_type'] = KG_file['Entity1_type'].str.lower()
KG_file['Entity2_type'] = KG_file['Entity2_type'].str.lower()
KG_file['Entity2_name'] = KG_file['Entity2_name'].str.lower()
KG_file['RELATION']     = KG_file['Entity1_type'] + '_' + KG_file['Entity2_type']

# Title-case the relation: 'gene_disease' -> 'Gene_Disease'
def format_relation(relation):
    """Capitalise the first letter of each part of a relation string split by '_'."""
    if isinstance(relation, str):
        return '_'.join([p.capitalize() for p in relation.split('_')])
    return relation

KG_file['RELATION'] = KG_file['RELATION'].apply(format_relation)

print("Relation distribution:")
print(KG_file['RELATION'].value_counts())

Relation distribution:
RELATION
Chemical_Disease     363230
Chemical_Gene        201206
Gene_Gene            190053
Gene_Disease         128028
Chemical_Chemical    111746
Gene_Chemical         98316
Disease_Disease         442
Disease_Gene            113
Disease_Chemical        102
Name: count, dtype: int64


In [13]:
# ── 2c. Rename columns to KG schema ──────────────────────────────────────────
KG_file = KG_file.rename(columns={
    'Entity1_name':     'Head',
    'Entity2_name':     'Tail',
    'Entity1_type':     'Head_type',
    'Entity2_type':     'Tail_type',
    'relationship_type':'Relation_type',
    'RELATION':         'Relation'
})
KG_file['KG_Source'] = 'PharmKG'

# and 'KG_Source' -> SyntaxError
desired_cols = [
    "Head", "Relation", "Tail", "Head_type", "Relation_type", "Tail_type",
    "Source", "PubMed_ID", "Sentence_tokenized", "KG_Source",
    "Head_ID", "Head_Name", "Head_Detail_name", "Head_ENS",
    "Tail_ID", "Tail_Name", "Tail_DOID_Name", "Tail_ENS",
    "Head_ID_IS", "Tail_ID_IS", "Head_Orignal", "Tail_Orignal"
]
KG_file = KG_file[[c for c in desired_cols if c in KG_file.columns]]
print(f"Schema applied. Columns retained: {list(KG_file.columns)}")

Schema applied. Columns retained: ['Head', 'Relation', 'Tail', 'Head_type', 'Relation_type', 'Tail_type', 'PubMed_ID', 'Sentence_tokenized', 'KG_Source']


In [14]:
# ── 2d. Split by Relation type into per-relation DataFrames ──────────────────
# Each relation type is saved as its own CSV file after ID normalisation below.
# We split here and store in a dict for clean access.
relation_dfs = {}
for relation in KG_file['Relation'].unique():
    relation_dfs[relation] = KG_file[KG_file['Relation'] == relation].copy()

print(f"Unique relations: {len(relation_dfs)}")
for rel, df in sorted(relation_dfs.items(), key=lambda x: -len(x[1])):
    print(f"  {rel:<30s}  {len(df):>7,} rows")

Unique relations: 9
  Chemical_Disease                363,230 rows
  Chemical_Gene                   201,206 rows
  Gene_Gene                       190,053 rows
  Gene_Disease                    128,028 rows
  Chemical_Chemical               111,746 rows
  Gene_Chemical                    98,316 rows
  Disease_Disease                     442 rows
  Disease_Gene                        113 rows
  Disease_Chemical                    102 rows


---
## 3 · Helper functions

In [15]:
def resolve_gene_head(df):
    """
    PharmKG stores gene names as full descriptions (e.g. 'peroxisome proliferator activated receptor gamma').
    Steps:
      1. Map full name -> Symbol via NCBI_fullname_2_SYMBOL_dict
      2. Uppercase (NCBI canonical symbols are uppercase for Human)
      3. Apply synonym map -> canonical Symbol
      4. Map Symbol -> description (Head_Detail_name)
      5. Swap: canonical Symbol -> Head, original full name -> Head_Ori (audit trail)
    """
    df = df.copy()
    df['Head_Ori'] = df['Head'].map(NCBI_fullname_2_SYMBOL_dict).fillna(df['Head'])
    df[['Head', 'Head_Ori']] = df[['Head_Ori', 'Head']]
    df['Head'] = df['Head'].str.upper()
    df['Head_Alias_mapped'] = df['Head'].apply(lambda x: synonym_map.get(x, x))
    df['Head_Detail_name']  = df['Head_Alias_mapped'].map(NCBI_ALL_GENEname_dict)
    df['Head_ID_IS'] = 'NCBI_ID'
    return df


def resolve_gene_tail(df):
    """Same as resolve_gene_head but applied to the Tail column."""
    df = df.copy()
    df['Tail_Ori'] = df['Tail'].map(NCBI_fullname_2_SYMBOL_dict).fillna(df['Tail'])
    df[['Tail', 'Tail_Ori']] = df[['Tail_Ori', 'Tail']]
    df['Tail'] = df['Tail'].str.upper()
    df['Tail_Alias_mapped'] = df['Tail'].apply(lambda x: synonym_map.get(x, x))
    df['Tail_Detail_name']  = df['Tail_Alias_mapped'].map(NCBI_ALL_GENEname_dict)
    df['Tail_ID_IS'] = 'NCBI_ID'
    return df


def resolve_disease(df, col):
    """
    Resolve disease name -> canonical ID via a 4-source cascade:
      1. DO label -> DOID (preferred)
      2. MeSH label -> MeSH xref (via Meshname2DOID_dict)
      3. MeSH ID -> name (via MESH_dict)
      4. MeSH supplementary ID -> name (via Mesh_supp_dict)
    Swaps resolved ID into col; original name moves to {col}_Orignal.
    Note: steps 2-4 may return MeSH IDs rather than DOIDs where DO coverage is missing.
    """
    df = df.copy()
    id_col   = f'{col}_ID_tmp'
    orig_col = f'{col}_Orignal'
    df[id_col] = (df[col].map(DOID_disname_DOID_dict)
                  .fillna(df[col].map(Meshname2DOID_dict))
                  .fillna(df[col].map(MESH_dict))
                  .fillna(df[col].map(Mesh_supp_dict)))
    before = len(df)
    df = df[df[id_col].notna()]
    print(f"  Disease {col} resolved: {len(df):,} kept / {before - len(df):,} dropped")
    df[orig_col] = df[col]        # save original name
    df[col]      = df[id_col]     # put resolved ID into col
    df.drop(columns=[id_col], inplace=True)
    df[f'{col}_ID_IS'] = 'DOID_ID'
    return df


def resolve_chemical(df, col):
    """
    Resolve chemical name -> PubChem CID via case-insensitive synonym lookup.
    Swaps CID into col; original name moves to {col}_Orignal.
    Rows without a resolved CID are dropped.
    """
    df = df.copy()
    id_col   = f'{col}_ID_tmp'
    orig_col = f'{col}_Orignal'
    df[id_col] = df[col].map(Pubchem_Syn_fil_dict_lower)
    df[id_col] = df[id_col].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', np.nan)
    before = len(df)
    df = df[df[id_col].notna()]
    print(f"  Chemical {col} -> PubChem CID: {len(df):,} kept / {before - len(df):,} dropped")
    df[orig_col] = df[col]
    df[col]      = df[id_col]
    df.drop(columns=[id_col], inplace=True)
    df[f'{col}_ID_IS'] = 'Pubchem'
    return df


def select_cols(df):
    """Select only columns present in df from the standard DESIRED_COLS list."""
    return df[[c for c in DESIRED_COLS if c in df.columns]]


def save(df, filename):
    """Save to OUT_PATH and print confirmation."""
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


---
## 4 · Process and export each relation type

### 4a · Gene_Gene

In [16]:
df_Gene_Gene = relation_dfs.get('Gene_Gene', pd.DataFrame()).copy()

df_Gene_Gene = resolve_gene_head(df_Gene_Gene)
df_Gene_Gene = resolve_gene_tail(df_Gene_Gene)

# Drop rows where either gene could not be validated against NCBI
before = len(df_Gene_Gene)
df_Gene_Gene = df_Gene_Gene[~df_Gene_Gene['Head_Detail_name'].isna()]
df_Gene_Gene = df_Gene_Gene[~df_Gene_Gene['Tail_Detail_name'].isna()]
print(f"NCBI validated: {len(df_Gene_Gene):,} kept / {before - len(df_Gene_Gene):,} dropped")

df_Gene_Gene['KG_Source'] = 'PharmKG'
df_Gene_Gene = select_cols(df_Gene_Gene)
save(df_Gene_Gene, 'PharmKG_Gene_Gene.csv')

NCBI validated: 102,072 kept / 87,981 dropped
Saved 102,072 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Gene_Gene.csv


### 4b · Gene_Disease

In [17]:
df_Gene_Disease = relation_dfs.get('Gene_Disease', pd.DataFrame()).copy()

df_Gene_Disease = resolve_gene_head(df_Gene_Disease)
before = len(df_Gene_Disease)
df_Gene_Disease = df_Gene_Disease[~df_Gene_Disease['Head_Detail_name'].isna()]
print(f"Gene Head NCBI validated: {len(df_Gene_Disease):,} kept / {before - len(df_Gene_Disease):,} dropped")

df_Gene_Disease = resolve_disease(df_Gene_Disease, 'Tail')
df_Gene_Disease['KG_Source'] = 'PharmKG'
df_Gene_Disease['Head_ID_IS'] = 'NCBI_ID'
df_Gene_Disease = select_cols(df_Gene_Disease)
save(df_Gene_Disease, 'PharmKG_Gene_Disease.csv')

Gene Head NCBI validated: 91,790 kept / 36,238 dropped
  Disease Tail resolved: 28,273 kept / 63,517 dropped
Saved 28,273 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Gene_Disease.csv


### 4c · Disease_Disease

In [18]:
df_Disease_Disease = relation_dfs.get('Disease_Disease', pd.DataFrame()).copy()

df_Disease_Disease = resolve_disease(df_Disease_Disease, 'Head')
df_Disease_Disease = resolve_disease(df_Disease_Disease, 'Tail')
df_Disease_Disease['KG_Source']  = 'PharmKG'
df_Disease_Disease['Head_ID_IS'] = 'DOID_ID'
df_Disease_Disease['Tail_ID_IS'] = 'DOID_ID'
df_Disease_Disease = select_cols(df_Disease_Disease)
save(df_Disease_Disease, 'PharmKG_Disease_Disease.csv')

  Disease Head resolved: 133 kept / 309 dropped
  Disease Tail resolved: 46 kept / 87 dropped
Saved 46 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Disease_Disease.csv


### 4d · Chemical_Gene

In [19]:
df_Chemical_Gene = relation_dfs.get('Chemical_Gene', pd.DataFrame()).copy()

# Resolve Chemical Head -> PubChem CID
df_Chemical_Gene = resolve_chemical(df_Chemical_Gene, 'Head')

# Resolve Gene Tail -> canonical symbol
df_Chemical_Gene = resolve_gene_tail(df_Chemical_Gene)
before = len(df_Chemical_Gene)
df_Chemical_Gene = df_Chemical_Gene[~df_Chemical_Gene['Tail_Detail_name'].isna()]
print(f"Gene Tail NCBI validated: {len(df_Chemical_Gene):,} kept / {before - len(df_Chemical_Gene):,} dropped")

df_Chemical_Gene['KG_Source'] = 'PharmKG'
df_Chemical_Gene = select_cols(df_Chemical_Gene)
save(df_Chemical_Gene, 'PharmKG_Chemical_Gene.csv')

  Chemical Head -> PubChem CID: 127,034 kept / 74,172 dropped
Gene Tail NCBI validated: 85,707 kept / 41,327 dropped
Saved 85,707 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Chemical_Gene.csv


### 4e · Chemical_Chemical

In [20]:
df_Chemical_Chemical = relation_dfs.get('Chemical_Chemical', pd.DataFrame()).copy()

df_Chemical_Chemical = resolve_chemical(df_Chemical_Chemical, 'Head')
df_Chemical_Chemical = resolve_chemical(df_Chemical_Chemical, 'Tail')
df_Chemical_Chemical['KG_Source']  = 'PharmKG'
df_Chemical_Chemical['Head_ID_IS'] = 'Pubchem'
df_Chemical_Chemical['Tail_ID_IS'] = 'Pubchem'
df_Chemical_Chemical = select_cols(df_Chemical_Chemical)
save(df_Chemical_Chemical, 'PharmKG_Chemical_Chemical.csv')

  Chemical Head -> PubChem CID: 73,562 kept / 38,184 dropped
  Chemical Tail -> PubChem CID: 33,325 kept / 40,237 dropped
Saved 33,325 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Chemical_Chemical.csv


### 4f · Disease_Chemical

In [21]:
df_Disease_Chemical = relation_dfs.get('Disease_Chemical', pd.DataFrame()).copy()

df_Disease_Chemical = resolve_disease(df_Disease_Chemical, 'Head')
# — renaming a column then immediately renaming the new name in the same call is silently ignored.
# resolve_chemical handles the swap cleanly with a single intermediate column.
df_Disease_Chemical = resolve_chemical(df_Disease_Chemical, 'Tail')
df_Disease_Chemical['KG_Source']  = 'PharmKG'
df_Disease_Chemical['Head_ID_IS'] = 'DOID'
df_Disease_Chemical['Tail_ID_IS'] = 'Pubchem'
df_Disease_Chemical = select_cols(df_Disease_Chemical)
save(df_Disease_Chemical, 'PharmKG_Disease_Chemical.csv')

  Disease Head resolved: 18 kept / 84 dropped
  Chemical Tail -> PubChem CID: 11 kept / 7 dropped
Saved 11 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Disease_Chemical.csv


### 4g · Disease_Gene

In [22]:
df_Disease_Gene = relation_dfs.get('Disease_Gene', pd.DataFrame()).copy()

df_Disease_Gene = resolve_disease(df_Disease_Gene, 'Head')
df_Disease_Gene = resolve_gene_tail(df_Disease_Gene)
before = len(df_Disease_Gene)
df_Disease_Gene = df_Disease_Gene[~df_Disease_Gene['Tail_Detail_name'].isna()]
print(f"Gene Tail NCBI validated: {len(df_Disease_Gene):,} kept / {before - len(df_Disease_Gene):,} dropped")

df_Disease_Gene['KG_Source']  = 'PharmKG'
df_Disease_Gene['Head_ID_IS'] = 'DOID'
df_Disease_Gene['Tail_ID_IS'] = 'NCBI_ID'
df_Disease_Gene = select_cols(df_Disease_Gene)
save(df_Disease_Gene, 'PharmKG_Disease_Gene.csv')

  Disease Head resolved: 32 kept / 81 dropped
Gene Tail NCBI validated: 22 kept / 10 dropped
Saved 22 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Disease_Gene.csv


### 4h · Gene_Chemical

In [23]:
df_Gene_Chemical = relation_dfs.get('Gene_Chemical', pd.DataFrame()).copy()

df_Gene_Chemical = resolve_gene_head(df_Gene_Chemical)
# AND no inplace/reassignment -> double-silent failure. Column is always 'Head_Detailed_name'
# from the loop; resolve_gene_head now consistently produces 'Head_Detail_name'.
before = len(df_Gene_Chemical)
df_Gene_Chemical = df_Gene_Chemical[~df_Gene_Chemical['Head_Detail_name'].isna()]
print(f"Gene Head NCBI validated: {len(df_Gene_Chemical):,} kept / {before - len(df_Gene_Chemical):,} dropped")

df_Gene_Chemical = resolve_chemical(df_Gene_Chemical, 'Tail')
df_Gene_Chemical['KG_Source']  = 'PharmKG'
df_Gene_Chemical['Head_ID_IS'] = 'NCBI_ID'
df_Gene_Chemical['Tail_ID_IS'] = 'Pubchem'
df_Gene_Chemical = select_cols(df_Gene_Chemical)
save(df_Gene_Chemical, 'PharmKG_Gene_Chemical.csv')

Gene Head NCBI validated: 76,888 kept / 21,428 dropped
  Chemical Tail -> PubChem CID: 30,895 kept / 45,993 dropped
Saved 30,895 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Gene_Chemical.csv


### 4i · Chemical_Disease

In [24]:
df_Chemical_Disease = relation_dfs.get('Chemical_Disease', pd.DataFrame()).copy()

df_Chemical_Disease = resolve_chemical(df_Chemical_Disease, 'Head')
df_Chemical_Disease = resolve_disease(df_Chemical_Disease, 'Tail')
df_Chemical_Disease['KG_Source']  = 'PharmKG'
df_Chemical_Disease['Head_ID_IS'] = 'Pubchem'
df_Chemical_Disease['Tail_ID_IS'] = 'DOID'
df_Chemical_Disease = select_cols(df_Chemical_Disease)
save(df_Chemical_Disease, 'PharmKG_Chemical_Disease.csv')

  Chemical Head -> PubChem CID: 227,480 kept / 135,750 dropped
  Disease Tail resolved: 70,487 kept / 156,993 dropped
Saved 70,487 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/PharmKG_Chemical_Disease.csv


---
## 5 · Summary — all output files

In [25]:
print(f"Output files under: {OUT_PATH}\n")
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]
df_list = []

for file_path in sorted(glob(os.path.join(OUT_PATH, '*.csv'))):
    try:
        total_df = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df = total_df.head(5)
        available = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.basename(file_path), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PharmKG/

Total files: 9


,Filename,no. of triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,PharmKG_Chemical_Chemical.csv,33325,{Chemical_Chemical},{chemical},{chemical},{PharmKG},{Pubchem},{Pubchem}
1,PharmKG_Chemical_Disease.csv,70487,{Chemical_Disease},{chemical},{disease},{PharmKG},{Pubchem},{DOID}
2,PharmKG_Chemical_Gene.csv,85707,{Chemical_Gene},{chemical},{gene},{PharmKG},{Pubchem},{NCBI_ID}
3,PharmKG_Disease_Chemical.csv,11,{Disease_Chemical},{disease},{chemical},{PharmKG},{DOID},{Pubchem}
4,PharmKG_Disease_Disease.csv,46,{Disease_Disease},{disease},{disease},{PharmKG},{DOID_ID},{DOID_ID}
5,PharmKG_Disease_Gene.csv,22,{Disease_Gene},{disease},{gene},{PharmKG},{DOID},{NCBI_ID}
6,PharmKG_Gene_Chemical.csv,30895,{Gene_Chemical},{gene},{chemical},{PharmKG},{NCBI_ID},{Pubchem}
7,PharmKG_Gene_Disease.csv,28273,{Gene_Disease},{gene},{disease},{PharmKG},{NCBI_ID},{DOID_ID}
8,PharmKG_Gene_Gene.csv,102072,{Gene_Gene},{gene},{gene},{PharmKG},{NCBI_ID},{NCBI_ID}
